# Colab 02 - Verify Qdrant Cloud

Use this notebook to verify that Qdrant Cloud collections are reachable and populated.

## 1. Clone / update project

This cell always lands in `/content/project-ks2` and verifies the repository layout.

In [ ]:
REPO_URL = "https://github.com/phamdinhhai/project-ks2.git"
PROJECT_DIR = "/content/project-ks2"

import os
from pathlib import Path

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git pull --ff-only || true

root = Path.cwd()
print("cwd:", root)
print("pyproject exists:", Path("pyproject.toml").exists())
print("medical_rag exists:", Path("src/medical_rag").exists())
assert Path("pyproject.toml").exists(), "Wrong folder: pyproject.toml not found"
assert Path("src/medical_rag").exists(), "Wrong folder: src/medical_rag not found"


## 2. Install dependencies

Editable install keeps local source changes active in Colab.

In [ ]:
!python -m pip install -U pip
!python -m pip install -e ".[qdrant,agent,eval]"
!python -m pip install -U requests huggingface_hub
!python -c "import medical_rag; print('medical_rag import OK')"


## 3. Configure secrets and model defaults

Secrets are read from Colab Secrets. BioMedBERT uses the correct public large checkpoint.

In [ ]:
import os

try:
    from google.colab import userdata
    for name in ["OPENROUTER_API_KEY", "QDRANT_URL", "QDRANT_API_KEY", "HF_TOKEN"]:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
except Exception as exc:
    print("Colab userdata unavailable:", exc)

os.environ.setdefault("OPENROUTER_MODEL", "google/gemini-2.5-flash")
os.environ.setdefault("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
os.environ.setdefault("BIOMEDBERT_MODEL", "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract")
os.environ.setdefault("BIOMEDBERT_DIM", "1024")

print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("OPENROUTER_MODEL:", os.environ.get("OPENROUTER_MODEL"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))
print("QDRANT_API_KEY set:", bool(os.environ.get("QDRANT_API_KEY")))
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("BIOMEDBERT_MODEL:", os.environ.get("BIOMEDBERT_MODEL"))
print("BIOMEDBERT_DIM:", os.environ.get("BIOMEDBERT_DIM"))

assert os.environ.get("QDRANT_URL"), "Missing QDRANT_URL in Colab Secrets"
assert os.environ.get("QDRANT_API_KEY"), "Missing QDRANT_API_KEY in Colab Secrets"


## Verify Qdrant Cloud

In [ ]:
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


## Optional local in-memory dry run

This does not require Qdrant Cloud and uses mock encoders.

In [ ]:
!python -m medical_rag build-qdrant-index --qdrant-url :memory: --limit 20 --use-mock-models
